# Espressività del PMA sotto DM — trimero ad anello

**Perché questo notebook.** Nel dimero, `analisi_rotazioni_PMA.ipynb` risolveva la domanda di design "dove mettere le rotazioni che rompono $M$" (1q vs 2q, parametro condiviso o indipendente) — una conclusione topology-indipendente, già ereditata correttamente per l'anello senza bisogno di riverificarla. La domanda **specifica dell'anello**, mai posta prima, è diversa: *quale famiglia ha davvero il potere espressivo giusto per rappresentare il ground state sotto DM?* Questo notebook gioca per l'anello lo stesso ruolo di `analisi_rotazioni_PMA.ipynb` per il dimero: un'indagine mirata che precede e informa `confronto_ansatz_entangler_trimero_anello.ipynb`.

**Obiettivo.** In `confronto_ansatz_entangler_trimero_anello.ipynb`, lo sweep standard
(2 restart) mostra che quasi tutte le famiglie PMA con 6+ parametri restano leggermente
sotto $\mathcal F=1$ con il DM acceso (Opzione B, $D=0.15$), tranne `PMA-2qC.K2`. Qui
isolo *dove* il residuo è peggiore e verifico, con ottimizzazione dedicata (non lo sweep
standard), se è un vero limite di espressività o un artefatto di convergenza — stessa
metodologia di `analisi_rotazioni_PMA.ipynb` (dimero) e della sez. 5 di
`confronto_ansatz_entangler_trimero_catena.ipynb`.

**Contesto.** Trimero ad anello isoscele, $J=1,\ J'=0.4$, $b_c=2J+J'=2.4$. Hamiltoniana
con DM Opzione B (unica compatibile con la simmetria $1\leftrightarrow2$ rotta in modo
uniforme, vedi `analisi_dm_trimero.pdf`):
$$H = J\,\boldsymbol\sigma_1\cdot\boldsymbol\sigma_2 + J'(\boldsymbol\sigma_2\cdot\boldsymbol\sigma_3+\boldsymbol\sigma_3\cdot\boldsymbol\sigma_1)
+ b\sum_i Z_i + D\sum_{\langle ij\rangle}(X_iZ_j-Z_iX_j),\qquad D=0.15.$$

**Risultato in una riga.** La famiglia "2q" (RBS su tutti e tre i legami, poi $R_y$
indipendenti) ha un **tetto strutturale vero** $\mathcal F\approx0.99831$ vicino a
$b=0.05$, identico per 6 e 9 parametri; la famiglia "1q" (meno efficiente a $D=0$) lo
supera e raggiunge $\mathcal F=1$ esatto, ma solo a 9 parametri.

**Nota sulla cache.** Come negli altri notebook di sweep del progetto, i risultati si
salvano incrementalmente in `plateau_dm_ring_cache.json`: se il file esiste già
(consegna corrente), l'esecuzione è quasi istantanea.

## 0. Setup

In [1]:
import os, json, time
import numpy as np
from scipy.optimize import minimize
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.quantum_info import Statevector

from trimer_ring_exact import (
    trimer_hamiltonian, exact_sweep, critical_field,
    ground_state_projector, trimer_hamiltonian_dm, exact_sweep_dm,
    ground_state_projector_dm,
)

BOND12, BOND23, BOND31 = (2, 1), (1, 0), (0, 2)
BONDS = [BOND12, BOND23, BOND31]

def rbs_block(qc, phi, q0, q1):
    sub = QuantumCircuit(2, name="RBS")
    sub.h(0); sub.h(1); sub.cz(0, 1)
    sub.ry(phi, 0); sub.ry(-phi, 1)
    sub.cz(0, 1); sub.h(0); sub.h(1)
    qc.append(sub.to_gate(label="RBS"), [q0, q1])

def pma_1q_trimer(nparam):
    p = ParameterVector("p", nparam)
    qc = QuantumCircuit(3)
    qc.x(0)
    i, bond_idx = 0, 0
    rbs_block(qc, p[i], *BONDS[bond_idx % 3]); i += 1; bond_idx += 1
    while i < nparam:
        qc.ry(p[i], 0); i += 1
        if i < nparam:
            rbs_block(qc, p[i], *BONDS[bond_idx % 3]); i += 1; bond_idx += 1
    return qc

def pma_2q_trimer_exact(nparam):
    p = ParameterVector("p", nparam)
    qc = QuantumCircuit(3)
    qc.x(0)
    rbs_block(qc, p[0], *BOND12)
    rbs_block(qc, p[1], *BOND23)
    rbs_block(qc, p[2], *BOND31)
    i = 3
    while i < nparam:
        qc.ry(p[i], 0); qc.ry(p[i + 1], 1); qc.ry(p[i + 2], 2)
        i += 3
    return qc

def pma_2q_trimer_cyclic(K):
    nparam = 6 * K
    p = ParameterVector("p", nparam)
    qc = QuantumCircuit(3)
    qc.x(0)
    idx = 0
    for _ in range(K):
        rbs_block(qc, p[idx], *BOND12); idx += 1
        rbs_block(qc, p[idx], *BOND23); idx += 1
        rbs_block(qc, p[idx], *BOND31); idx += 1
        qc.ry(p[idx], 0); idx += 1
        qc.ry(p[idx], 1); idx += 1
        qc.ry(p[idx], 2); idx += 1
    return qc

CANDIDATI = {
    "PMA-2qC.K1": pma_2q_trimer_cyclic(1),
    "PMA-2q.6":   pma_2q_trimer_exact(6),
    "PMA-2q.9":   pma_2q_trimer_exact(9),
    "PMA-1q.6":   pma_1q_trimer(6),
    "PMA-1q.9":   pma_1q_trimer(9),
}
for name, qc in CANDIDATI.items():
    print(f"  {name:14s} {qc.num_parameters} parametri")

CACHE_PATH = "plateau_dm_ring_cache.json"
def load_cache():
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH) as f:
            return json.load(f)
    return {}
def save_cache(c):
    with open(CACHE_PATH, "w") as f:
        json.dump(c, f, indent=1)
cache = load_cache()
print(f"punti gia' in cache: {len(cache)}")

  PMA-2qC.K1     6 parametri
  PMA-2q.6       6 parametri
  PMA-2q.9       9 parametri
  PMA-1q.6       6 parametri
  PMA-1q.9       9 parametri
punti gia' in cache: 10


## 1. Il problema: cosa emerge dallo sweep standard

In `confronto_ansatz_entangler_trimero_unico.ipynb`, con lo sweep standard a 2 restart
(15 punti in $b$, $D=0.15$ Opzione B), la tabella riassuntiva mostra: *"da 6 parametri in
su, tutte le famiglie PMA raggiungono $\mathcal F=1.0000$ esatto a $D=0$. Con il DM
acceso, però, quasi tutte restano leggermente sotto 1 (minimo $\approx0.995$–$0.998$) —
tranne `PMA-2qC.K2`."* Il notebook non approfondisce oltre. Qui isolo dove il residuo è
peggiore e verifico se è strutturale.

## 2. Dove cade il minimo (grid search, 2 restart come nello sweep originale)

In [2]:
def _energy(params, ansatz, hamiltonian, estimator):
    pub = (ansatz, hamiltonian, [params])
    return float(estimator.run([pub]).result()[0].data.evs.item())

def run_vqe_ansatz(ansatz, J, Jp, b, n_restarts=2, seed=42, dm_mode=None, D=0.0):
    rng = np.random.default_rng(seed)
    hamiltonian = trimer_hamiltonian_dm(J, Jp, b, dm_mode, D)
    estimator = StatevectorEstimator()
    n_params = ansatz.num_parameters
    P0, _, deg = ground_state_projector_dm(J, Jp, b, dm_mode, D)
    best_e, best_params = np.inf, None
    for _ in range(n_restarts):
        x0 = rng.uniform(-np.pi, np.pi, n_params)
        r1 = minimize(_energy, x0, args=(ansatz, hamiltonian, estimator),
                      method="COBYLA", options={"maxiter": 2000, "tol": 1e-10})
        r2 = minimize(_energy, r1.x, args=(ansatz, hamiltonian, estimator),
                      method="L-BFGS-B", options={"maxiter": 2000, "ftol": 1e-14})
        if r2.fun < best_e:
            best_e, best_params = r2.fun, r2.x
    sv = Statevector(ansatz.assign_parameters(best_params)).data
    fid = float(np.real(sv.conj() @ P0 @ sv))
    return {"fidelity": fid, "n_params": n_params, "degeneracy": deg}

J, Jp = 1.0, 0.4
bc = critical_field(J, Jp)
B_GRID = sorted(set(np.linspace(0.05, 4.8, 15).round(3).tolist() + [2.4]))
print(f"b_c = {bc}, griglia: {B_GRID}")

t0 = time.time()
for name, ansatz in CANDIDATI.items():
    gridkey = f"gridmin|{name}"
    if gridkey in cache:
        print(f"{name:14s} min_fid={cache[gridkey]['fid_min']:.6f} a b={cache[gridkey]['b_min']}  (da cache)")
        continue
    fids = []
    for b in B_GRID:
        key = f"grid|{name}|{b}"
        if key in cache:
            fids.append(cache[key]["fidelity"])
            continue
        r = run_vqe_ansatz(ansatz, J, Jp, b, n_restarts=2, dm_mode="B", D=0.15)
        cache[key] = r
        save_cache(cache)
        fids.append(r["fidelity"])
    fids = np.array(fids)
    imin = np.argmin(fids)
    cache[gridkey] = {"b_min": float(B_GRID[imin]), "fid_min": float(fids[imin])}
    save_cache(cache)
    print(f"{name:14s} min_fid={fids[imin]:.6f} a b={B_GRID[imin]}")
print(f"tempo: {time.time()-t0:.1f}s")

b_c = 2.4, griglia: [0.05, 0.389, 0.729, 1.068, 1.407, 1.746, 2.086, 2.4, 2.425, 2.764, 3.104, 3.443, 3.782, 4.121, 4.461, 4.8]
PMA-2qC.K1     min_fid=0.998128 a b=0.05  (da cache)
PMA-2q.6       min_fid=0.998128 a b=0.05  (da cache)
PMA-2q.9       min_fid=0.998128 a b=0.05  (da cache)
PMA-1q.6       min_fid=0.996224 a b=0.05  (da cache)
PMA-1q.9       min_fid=0.994828 a b=0.05  (da cache)
tempo: 0.0s


**Osservazione.** Il minimo non cade vicino a $b_c=2.4$ (dove ci si aspetterebbe il
mescolamento massimo fra i blocchi $A$ e $C$), ma a $b=0.05$, quasi a campo nullo. Tre
varianti diverse (`PMA-2qC.K1`, `PMA-2q.6`, `PMA-2q.9`) toccano lì lo **stesso identico
valore** nello sweep standard — il primo segnale che non è rumore.

## 3. Verifica dedicata: ottimizzazione diretta della fidelity, 60 restart

In [3]:
def max_fidelity(qc, P0, n_restarts, seed=7):
    def neg_fid(params):
        sv = Statevector(qc.assign_parameters(params)).data
        return -np.real(sv.conj() @ P0 @ sv)
    rng = np.random.default_rng(seed)
    best = 0.0
    for _ in range(n_restarts):
        x0 = rng.uniform(-np.pi, np.pi, qc.num_parameters)
        r = minimize(neg_fid, x0, method="COBYLA", options={"maxiter": 800, "tol": 1e-13})
        r2 = minimize(neg_fid, r.x, method="L-BFGS-B", options={"maxiter": 800, "ftol": 1e-16})
        best = max(best, -r2.fun)
    return best

b_worst = 0.05
P0, E0, deg = ground_state_projector_dm(J, Jp, b_worst, "B", 0.15)
print(f"degenerazione del fondamentale a b={b_worst}: {deg}")

risultati = {}
for name, qc in CANDIDATI.items():
    key = f"fid60|{name}"
    if key in cache:
        risultati[name] = cache[key]["fidelity"]
        print(f"{name:14s} ({qc.num_parameters} par): "
              f"max fidelity su 60 restart = {cache[key]['fidelity']:.8f}  (da cache)")
        continue
    t0 = time.time()
    best = max_fidelity(qc, P0, n_restarts=60)
    risultati[name] = best
    cache[key] = {"fidelity": best, "n_restarts": 60}
    save_cache(cache)
    print(f"{name:14s} ({qc.num_parameters} par): "
          f"max fidelity su 60 restart = {best:.8f}  ({time.time()-t0:.1f}s)")

degenerazione del fondamentale a b=0.05: 1
PMA-2qC.K1     (6 par): max fidelity su 60 restart = 0.99830708  (da cache)
PMA-2q.6       (6 par): max fidelity su 60 restart = 0.99830708  (da cache)
PMA-2q.9       (9 par): max fidelity su 60 restart = 0.99830708  (da cache)
PMA-1q.6       (6 par): max fidelity su 60 restart = 0.99661662  (da cache)
PMA-1q.9       (9 par): max fidelity su 60 restart = 1.00000000  (da cache)


## 4. Risultato

| Ansatz | Parametri | Fidelity max (60 restart) |
|---|---|---|
| `PMA-2qC.K1` | 6 | $0.99830708$ |
| `PMA-2q.6` | 6 | $0.99830708$ |
| `PMA-2q.9` | 9 | $0.99830708$ — identico, 3 parametri in più non aiutano |
| `PMA-1q.6` | 6 | $0.99661662$ — peggiore della famiglia 2q |
| `PMA-1q.9` | 9 | $\mathbf{1.00000000}$ — esatto |

Confermato con 60 restart (30 volte più dei 2 dello sweep originale) e ottimizzazione
diretta della fidelity (non dell'energia, per escludere che il problema sia solo la
funzione di costo usata dal VQE): tre varianti indipendenti della famiglia "2q"
convergono **esattamente allo stesso valore**, $0.99830708$, sia a 6 sia a 9 parametri.
Non è un problema di convergenza — è un tetto di espressività strutturale di quella
particolare architettura (un solo giro iniziale di RBS su tutti e tre i legami, poi
$R_y$ indipendenti a triple), in questo specifico regime fisico.

## 5. Lettura: il risultato è controintuitivo rispetto a $D=0$

A $D=0$ la famiglia "2q" è strettamente migliore della "1q" (raggiunge $\mathcal F=1$ da
3 parametri contro i 4 della "1q", vedi `analisi_rotazioni_PMA.ipynb`): più gradi di
libertà indipendenti per blocco, meno ridondanza. **Qui, sotto DM e vicino a $b=0$,
succede l'opposto**: la "2q" resta bloccata a $\approx0.9983$ qualunque sia il numero di
parametri (nello schema testato), mentre la "1q" — meno efficiente in generale — arriva
all'esatto con 9 parametri.

**Perché non è contraddittorio.** Le due famiglie non esplorano lo stesso sottospazio
via via che si aggiungono parametri: la "2q" ripete sempre la stessa struttura (un giro
di RBS su tutti i legami, poi triple di $R_y$ indipendenti — bloccata nel sottospazio
raggiungibile da quella particolare sequenza di generatori), mentre la "1q" alterna un
singolo blocco RBS con singole rotazioni locali cicliche sui tre legami: una sequenza
diversa di generatori non commutanti può raggiungere settori dello spazio di Hilbert
preclusi all'altra, indipendentemente da quale delle due abbia più parametri in totale.
Il numero di parametri misura la dimensione locale del sottospazio esplorato, non
garantisce che quel sottospazio contenga il vero ground state.

**Implicazione pratica.** La scelta canonica per l'anello non può basarsi solo sul
risultato a $D=0$ (dove la "2q" vince) né sul solo conteggio di gate: se il termine DM
resta nello scope della tesi, la famiglia "1q" a sufficienti parametri (9, non 6) va
tenuta come alternativa concreta vicino a $b=0$, non scartata a priori. Da verificare se
lo stesso schema si ripete ad altri valori di $D$ e in altri punti della griglia, prima
di aggiornare la raccomandazione finale in `confronto_ansatz_entangler_trimero_unico.ipynb`.